# Next Day Wildfire Spread

Each sample is a 64 by 64 km patch at 1 km resolution. Twelve grids are inputs, and `FireMask` is the label: where fire is the next day. The fire masks use 1 for fire, 0 for no fire, and -1 for uncertain, usually clouds.

This notebook reads the first data file only. The loading code lives in `inspect_ndws.py` next to this notebook. It downloads that file into `data/ndws/` at the repo root the first time it runs.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from inspect_ndws import download_first_shard, feature_table, read_examples, sample_table

SAMPLE_COUNT = 20

shard = download_first_shard()
examples = read_examples(shard, SAMPLE_COUNT)
print(f"Read {len(examples)} samples from {shard.name}")

## Features

One row per grid. `min`, `max`, and `mean` are over every pixel in the samples loaded above.

In [ ]:
features = feature_table(examples)
features

## Samples

One row per patch: the mean of each grid, plus how many pixels of yesterday's and tomorrow's fire masks are fire, no fire, or uncertain.

In [ ]:
samples = sample_table(examples)
samples

## Yesterday's fire next to tomorrow's fire

Change `SAMPLE` to look at a different patch. Red is fire, light grey is no fire, and dark grey is uncertain.

In [ ]:
SAMPLE = 1

mask_colors = ListedColormap(["dimgray", "whitesmoke", "red"])
example = examples[SAMPLE]

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, name, title in zip(
    axes,
    ["PrevFireMask", "FireMask"],
    ["Yesterday's fire (input)", "Tomorrow's fire (label)"],
):
    ax.imshow(example[name], cmap=mask_colors, vmin=-1, vmax=1, interpolation="nearest")
    fire_pixels = int((example[name] == 1).sum())
    ax.set_title(f"{title}\n{fire_pixels} fire pixels")
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f"Sample {SAMPLE}")
plt.tight_layout()

## All inputs for one sample

Each panel is one input grid for the same patch, with its own color scale.

In [ ]:
input_names = features.loc[features["role"] == "input", "feature"].tolist()

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, name in zip(axes.flat, input_names):
    grid = example[name]
    if name == "PrevFireMask":
        image = ax.imshow(grid, cmap=mask_colors, vmin=-1, vmax=1, interpolation="nearest")
    else:
        image = ax.imshow(grid, cmap="viridis")
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
for ax in axes.flat[len(input_names):]:
    ax.axis("off")
fig.suptitle(f"Inputs for sample {SAMPLE}")
plt.tight_layout()